[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/15_cross_validation/02_group_timeseries_loo.ipynb)

# Bài 02 — Group K-Fold, Time Series Split và Leave-One-Out

**🎯 Mục tiêu**
- Nhận ra khi các mẫu **không độc lập**: nhiều mẫu của cùng một bệnh nhân / khách hàng / thiết bị → **GroupKFold**.
- Dữ liệu theo **thời gian**: train phải nằm **trước** validation → `TimeSeriesSplit` (expanding/sliding window, `gap`).
- So sánh ước lượng CV với hiệu năng "thật" trong tình huống triển khai — CV sai cách lạc quan đến mức nào.
- Leave-One-Out: khi nào dùng, chi phí và variance.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/15_cross_validation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold, KFold, LeaveOneOut, TimeSeriesSplit, cross_val_score

from ml_course.datasets import make_sales_series
from ml_course.validation import GroupKFoldScratch, TimeSeriesSplitScratch
from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Dữ liệu có nhóm: nhiều mẫu mỗi bệnh nhân

60 bệnh nhân, mỗi người đo 20 lần. Mỗi người có một "dấu vân tay" sinh học riêng (5 chỉ số nền khác nhau giữa người với người), và bệnh làm chỉ số đầu tiên tăng nhẹ. Nhãn ở **cấp bệnh nhân**. Câu hỏi thật: model có chẩn đoán đúng **bệnh nhân mới** không?

In [ ]:
n_patients, n_visits = 60, 20
fingerprint = rng.normal(0, 1, (n_patients, 5))
sick = rng.random(n_patients) < 0.5
groups = np.repeat(np.arange(n_patients), n_visits)
Xg = fingerprint[groups] + 0.8 * sick[groups][:, None] * np.array([1, 0, 0, 0, 0]) + rng.normal(0, 0.3, (len(groups), 5))
yg = sick[groups].astype(int)

# "sự thật": 60 bệnh nhân hoàn toàn mới, sinh từ cùng quy luật
fp_new, sick_new = rng.normal(0, 1, (60, 5)), rng.random(60) < 0.5
g_new = np.repeat(np.arange(60), n_visits)
X_new = fp_new[g_new] + 0.8 * sick_new[g_new][:, None] * np.array([1, 0, 0, 0, 0]) + rng.normal(0, 0.3, (len(g_new), 5))
y_new = sick_new[g_new].astype(int)

clf = RandomForestClassifier(n_estimators=200, random_state=0)
random_cv = cross_val_score(clf, Xg, yg, cv=KFold(5, shuffle=True, random_state=0)).mean()
group_cv = cross_val_score(clf, Xg, yg, cv=GroupKFold(5), groups=groups).mean()
true_acc = clf.fit(Xg, yg).score(X_new, y_new)
print(f"KFold ngẫu nhiên    : accuracy = {random_cv:.3f}")
print(f"GroupKFold          : accuracy = {group_cv:.3f}")
print(f"Bệnh nhân mới (thật): accuracy = {true_acc:.3f}")

KFold ngẫu nhiên đặt các lần đo **của cùng một người** vào cả train và validation → model chỉ cần **nhận ra người đó** (qua dấu vân tay) chứ không cần học dấu hiệu bệnh → ước lượng lạc quan khủng khiếp. GroupKFold đặt **mọi** mẫu của một người vào cùng một fold → khớp với hiệu năng thật trên bệnh nhân mới.

Ví dụ nhóm trong thực tế: bệnh nhân, khách hàng (nhiều giao dịch), người dùng (nhiều phiên), cửa hàng, thiết bị/cảm biến, ảnh từ cùng một video, câu từ cùng một tài liệu, nhà cùng một toà chung cư.

In [ ]:
ours = [np.unique(groups[val]).size for _, val in GroupKFoldScratch(5).split(Xg, yg, groups)]
leak = [len(np.intersect1d(groups[tr], groups[val])) for tr, val in GroupKFoldScratch(5).split(Xg, yg, groups)]
print("GroupKFoldScratch: số bệnh nhân trong mỗi fold validation:", ours, "| số bệnh nhân xuất hiện ở cả train lẫn val:", leak)

## 2. Dữ liệu theo thời gian

Dự báo doanh số ngày. Feature: chỉ số ngày (xu hướng), thứ trong tuần, ngày trong năm, khuyến mãi, ngày lễ. Câu hỏi thật: model dự báo **tương lai** tốt đến đâu?

In [ ]:
sales = make_sales_series(n_days=3 * 365, seed=1)
feat = pd.DataFrame({"t": np.arange(len(sales)), "dow": sales.date.dt.dayofweek, "doy": sales.date.dt.dayofyear,
                     "promo": sales.promo, "holiday": sales.is_holiday})
target = sales.sales.to_numpy()
cut = int(0.8 * len(sales))
X_hist, y_hist, X_future, y_future = feat.iloc[:cut], target[:cut], feat.iloc[cut:], target[cut:]

reg = HistGradientBoostingRegressor(random_state=0)
mae_random = -cross_val_score(reg, X_hist, y_hist, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error").mean()
mae_ts = -cross_val_score(reg, X_hist, y_hist, cv=TimeSeriesSplit(5, test_size=90), scoring="neg_mean_absolute_error").mean()
mae_future = mean_absolute_error(y_future, reg.fit(X_hist, y_hist).predict(X_future))
print(f"KFold ngẫu nhiên      : MAE = {mae_random:.1f}")
print(f"TimeSeriesSplit       : MAE = {mae_ts:.1f}")
print(f"Tương lai thật (20%)  : MAE = {mae_future:.1f}")

KFold ngẫu nhiên cho phép model "nội suy" giữa các ngày lân cận đã thấy (hôm qua và ngày mai đều ở train) → dễ hơn nhiều so với dự báo tương lai thật, nơi xu hướng tiếp tục tăng mà cây không ngoại suy được → **lạc quan**. `TimeSeriesSplit` luôn validate trên khoảng thời gian **sau** train → ước lượng gần thực tế hơn, và lệch về phía **bi quan** (các split đầu chỉ train trên ít dữ liệu) — lệch về phía an toàn.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
n_show = len(X_hist)
for ax, (name, cv) in zip(axes, [("expanding window", TimeSeriesSplit(5, test_size=90)),
                                 ("sliding window (max_train_size=365, gap=7)", TimeSeriesSplit(5, test_size=90, max_train_size=365, gap=7))]):
    for i, (tr, va) in enumerate(cv.split(X_hist)):
        ax.plot(tr, np.full(len(tr), i), color="tab:blue", lw=6, solid_capstyle="butt")
        ax.plot(va, np.full(len(va), i), color="tab:red", lw=6, solid_capstyle="butt")
    ax.set_title(f"TimeSeriesSplit — {name}: xanh = train, đỏ = validation")
    ax.set_yticks(range(5), [f"split {i + 1}" for i in range(5)])
axes[1].set_xlabel("ngày")
plt.tight_layout()
plt.show()

- **Expanding window**: train tích luỹ toàn bộ quá khứ. **Sliding window** (`max_train_size`): chỉ dùng $W$ ngày gần nhất — khi quy luật thay đổi theo thời gian.
- **`gap`**: bỏ khoảng trống giữa train và validation — mô phỏng độ trễ dữ liệu (vd dữ liệu hôm nay chỉ có sau 7 ngày) và tránh rò rỉ qua feature lag/rolling.

In [ ]:
ours_ts = [(tr.max(), va.min(), va.max()) for tr, va in TimeSeriesSplitScratch(5, test_size=90, gap=7).split(X_hist)]
print("TimeSeriesSplitScratch (train kết thúc, val bắt đầu, val kết thúc):", ours_ts)

## 3. Leave-One-Out (LOO)

$k = n$: mỗi lần bỏ ra **một** mẫu. Ưu điểm: dùng gần hết dữ liệu để train (ít bi quan) — hữu ích khi dữ liệu **rất ít**. Nhược điểm: $n$ lần train (đắt), và ước lượng có **variance cao** (các model gần như giống hệt nhau → lỗi tương quan mạnh).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier

Xi, yi = load_iris(return_X_y=True)
loo = cross_val_score(KNeighborsClassifier(5), Xi, yi, cv=LeaveOneOut())
print(f"LOO: {len(loo)} lần train, accuracy = {loo.mean():.4f} (mỗi lần chỉ là 0 hoặc 1)")
for k in [5, 10]:
    s = [cross_val_score(KNeighborsClassifier(5), Xi, yi, cv=KFold(k, shuffle=True, random_state=r)).mean() for r in range(20)]
    print(f"{k}-fold (20 cách xáo trộn): {np.mean(s):.4f} ± {np.std(s):.4f}")

Với Linear Regression / Ridge, LOO có **công thức đóng** (không cần train lại $n$ lần): $e_{(i)} = \frac{e_i}{1 - h_{ii}}$ với $h_{ii}$ là leverage — `RidgeCV` dùng điều này để chọn $\alpha$ cực nhanh.

## 4. Chọn chiến lược CV

| Tình huống | Chiến lược |
|---|---|
| Mẫu độc lập, phân loại | `StratifiedKFold` (5–10) |
| Mẫu độc lập, hồi quy | `KFold` (5–10), có thể stratify theo bin của y |
| Nhiều mẫu mỗi thực thể (bệnh nhân, khách hàng) | `GroupKFold` / `StratifiedGroupKFold` |
| Dữ liệu theo thời gian / dự báo | `TimeSeriesSplit` (+ `gap`, sliding window) |
| Dữ liệu rất ít (< 100) | Repeated K-Fold, LOO |
| Dữ liệu không gian (bản đồ) | Spatial/block CV — chia theo vùng địa lý |
| Dữ liệu rất lớn | một lần hold-out lớn là đủ |

**Nguyên tắc vàng:** cách chia validation phải **mô phỏng đúng** cách model sẽ được dùng — model gặp bệnh nhân mới? tương lai? cửa hàng mới?

## 🧠 Kiểm tra nhanh

1. Dữ liệu 10 000 giao dịch từ 500 khách hàng; model dự đoán cho **khách hàng mới**. CV nào? Nếu dự đoán giao dịch tiếp theo của **khách hàng cũ**?
2. Vì sao KFold ngẫu nhiên lạc quan với dữ liệu chuỗi thời gian có xu hướng?
3. `gap` trong `TimeSeriesSplit` dùng để làm gì?
4. Vì sao LOO có variance cao dù dùng gần hết dữ liệu để train?

## 📌 Tóm tắt

- Mẫu không độc lập → GroupKFold; dữ liệu thời gian → TimeSeriesSplit; CV sai cách cho ước lượng lạc quan tới mức vô nghĩa.
- Chiến lược CV phải mô phỏng tình huống triển khai.
- LOO cho dữ liệu rất ít; đắt và variance cao.

## 📚 Tài liệu tham khảo

- Roberts et al. (2017) — *Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure*
- Bergmeir & Benítez (2012) — *On the use of cross-validation for time series predictor evaluation*
- scikit-learn User Guide — Cross-validation iterators for grouped / time series data